# Trabalho Prático 1 - INF01017
## Pré-processamento de Dados
### Dataset: Ames Mutagenicity

**Objetivo:** Preparar os dados para modelagem através de técnicas de pré-processamento.

---

## 1. Imports e Configurações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

# Configurações
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
import os

# Criar diretório para salvar figuras
os.makedirs('../results/figures/preprocessing/', exist_ok=True)
print('✓ Diretório de figuras criado!')

✓ Diretório de figuras criado!


## 2. Carregamento dos Dados

In [2]:
# Carregar dataset
df = pd.read_csv('../data/raw/ames_mutagenicity_data.csv')

print(f"Shape original: {df.shape}")
df.head()

Shape original: (5536, 1371)


,Id,Name,CAS,SMILES RDKit,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,...,Zagreb2,mZagreb1,mZagreb2,TA98,TA100,TA102,TA1535,TA1537,Overall,Partition
0,1,BENZO[A]PYRENE,50-32-8,c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34,0.364671,0.093588,-0.313685,-0.268813,0.423577,0.895798,...,0.588244,-0.520336,0.123180,1,1,-1,-1,1,1,Train
1,2,PIVALOLACTONE,1955-45-9,CC1(C)COC1=O,-0.811928,-0.820862,-0.313685,-0.268813,-0.884375,0.146261,...,-0.765222,-0.565069,-0.891039,0,1,-1,0,0,1,Train
2,3,"4,4'-METHYLENEBIS(N,N-DIMETHYLBENZENAMINE)",101-61-1,CN(C)c1ccc(Cc2ccc(N(C)C)cc2)cc1,0.138224,0.064344,-0.313685,-0.268813,0.133040,-0.097471,...,0.034553,0.108807,0.133477,0,0,-1,0,0,-1,Train
3,4,N-NITROSO-N-ETHYLANILINE,612-64-6,CCN(N=O)c1ccccc1,-0.561301,-0.517866,-0.313685,-0.268813,-0.497803,-0.381972,...,-0.617571,-0.503020,-0.401949,0,0,-1,0,0,-1,Internal
4,5,MICHLER'S KETONE,90-94-8,CN(C)c1ccc(C(=O)c2ccc(N(C)C)cc2)cc1,0.214333,0.196807,-0.313685,-0.268813,0.228942,0.079164,...,0.145291,0.287737,0.215850,0,0,-1,0,0,-1,Internal


## 3. Remoção de Colunas de Metadados

In [3]:
# Remover colunas de metadados
metadata_cols = ['Id', 'Name', 'CAS', 'SMILES RDKit']
cols_to_remove = [col for col in metadata_cols if col in df.columns]

print(f"Removendo colunas: {cols_to_remove}")
df_clean = df.drop(columns=cols_to_remove)

print(f"Shape após remoção: {df_clean.shape}")

Removendo colunas: ['Id', 'Name', 'CAS', 'SMILES RDKit']
Shape após remoção: (5536, 1367)


## 4. Identificação de Colunas

In [4]:
# Identificar tipos de colunas
target_col = 'Overall'
strain_cols = ['TA98', 'TA100', 'TA102', 'TA1535', 'TA1537']
partition_col = 'Partition'

# Opcionalmente, remover coluna de partição
if partition_col in df_clean.columns:
    df_clean = df_clean.drop(columns=[partition_col])
    print(f"Coluna '{partition_col}' removida.")

# Features
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

print(f"\nNúmero de features: {len(feature_cols)}")
print(f"Coluna alvo: {target_col}")

Coluna 'Partition' removida.

Número de features: 1360
Coluna alvo: Overall


## 5. Tratamento de Valores Faltantes

In [5]:
# Verificar valores faltantes
n_missing = df_clean.isnull().sum().sum()
print(f"Total de valores faltantes: {n_missing}")

if n_missing > 0:
    # Estratégia: remover colunas com >50% missing e linhas restantes
    threshold = 0.5
    missing_ratio = df_clean.isnull().sum() / len(df_clean)
    cols_to_drop = missing_ratio[missing_ratio > threshold].index.tolist()
    
    if cols_to_drop:
        df_clean = df_clean.drop(columns=cols_to_drop)
        print(f"Colunas removidas (>{threshold*100}% missing): {len(cols_to_drop)}")
    
    df_clean = df_clean.dropna()
    print(f"Shape após tratamento: {df_clean.shape}")
else:
    print("✓ Nenhum valor faltante encontrado!")

Total de valores faltantes: 0
✓ Nenhum valor faltante encontrado!


## 6. Remoção de Features com Baixa Variância

In [6]:
# Atualizar lista de features
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

# Calcular variância
threshold_variance = 0.01
variances = df_clean[feature_cols].var()
low_variance_features = variances[variances < threshold_variance].index.tolist()

print(f"Features com variância < {threshold_variance}: {len(low_variance_features)}")

if low_variance_features:
    df_clean = df_clean.drop(columns=low_variance_features)
    print(f"Shape após remoção: {df_clean.shape}")

Features com variância < 0.01: 6
Shape após remoção: (5536, 1360)


## 7. Normalização das Features

In [7]:
# Atualizar lista de features novamente
feature_cols = [col for col in df_clean.columns 
               if col not in [target_col] + strain_cols]

# Normalização usando StandardScaler
scaler = StandardScaler()
df_normalized = df_clean.copy()

numeric_features = [col for col in feature_cols if col in df_clean.columns 
                   and df_clean[col].dtype in [np.float64, np.int64]]

df_normalized[numeric_features] = scaler.fit_transform(df_clean[numeric_features])

print(f"Features normalizadas: {len(numeric_features)}")
print("\nEstatísticas após normalização (primeiras 5 features):")
df_normalized[numeric_features[:5]].describe()

Features normalizadas: 1354

Estatísticas após normalização (primeiras 5 features):


,ABC,ABCGG,nAcid,nBase,SpAbs_A
count,5.536000e+03,5536.000000,5.536000e+03,5.536000e+03,5.536000e+03
mean,5.133979e-18,0.000000,-5.133979e-17,-1.155145e-17,-3.080388e-17
std,1.000090e+00,1.000090,1.000090e+00,1.000090e+00,1.000090e+00
min,-1.333241e+00,-1.573058,-2.957354e-01,-2.623455e-01,-1.366655e+00
25%,-5.242210e-01,-0.503046,-2.957354e-01,-2.623455e-01,-5.344314e-01
50%,-1.010525e-01,-0.078725,-2.957354e-01,-2.623455e-01,-8.460811e-02
75%,3.487008e-01,0.326269,-2.957354e-01,-2.623455e-01,3.534818e-01
max,3.156659e+01,31.459452,1.852255e+01,2.475113e+01,3.197065e+01


## 8. Divisão dos Dados

In [8]:
# Separar features e target
X = df_normalized.drop(columns=[target_col])
y = df_normalized[target_col]

# Split: 70% treino, 10% validação, 20% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Tamanho do conjunto de treino: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Tamanho do conjunto de validação: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Tamanho do conjunto de teste: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print("\nDistribuição das classes:")
print(f"Treino: {y_train.value_counts().to_dict()}")
print(f"Validação: {y_val.value_counts().to_dict()}")
print(f"Teste: {y_test.value_counts().to_dict()}")

Tamanho do conjunto de treino: 3874 (70.0%)
Tamanho do conjunto de validação: 554 (10.0%)
Tamanho do conjunto de teste: 1108 (20.0%)

Distribuição das classes:
Treino: {1: 2171, -1: 1541, 0: 162}
Validação: {1: 311, -1: 220, 0: 23}
Teste: {1: 621, -1: 441, 0: 46}


## 9. Salvamento dos Dados Processados

In [9]:
import os

# Criar DataFrames completos
train_df = X_train.copy()
train_df[target_col] = y_train

val_df = X_val.copy()
val_df[target_col] = y_val

test_df = X_test.copy()
test_df[target_col] = y_test

# Salvar
output_path = 'data/processed/'
os.makedirs(output_path, exist_ok=True)

train_df.to_csv(os.path.join(output_path, 'train.csv'), index=False)
val_df.to_csv(os.path.join(output_path, 'validation.csv'), index=False)
test_df.to_csv(os.path.join(output_path, 'test.csv'), index=False)

print(f"Dados processados salvos em: {output_path}")
print("✓ Arquivos criados: train.csv, validation.csv, test.csv")

Dados processados salvos em: data/processed/
✓ Arquivos criados: train.csv, validation.csv, test.csv


In [10]:
# Salvar resumo do pré-processamento
import os
os.makedirs('../results/metrics/', exist_ok=True)

summary_path = '../results/metrics/preprocessing_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('='*60 + '\n')
    f.write('RESUMO DO PRÉ-PROCESSAMENTO\n')
    f.write('Dataset: Ames Mutagenicity\n')
    f.write('='*60 + '\n\n')
    
    f.write('1. ETAPAS REALIZADAS\n')
    f.write('   a) Remoção de colunas de metadados\n')
    f.write('   b) Tratamento de valores faltantes\n')
    f.write('   c) Remoção de features com baixa variância\n')
    f.write('   d) Normalização (StandardScaler)\n')
    f.write('   e) Divisão treino/validação/teste\n\n')
    
    f.write('2. DIMENSÕES FINAIS\n')
    f.write(f'   - Features: {X.shape[1]}\n')
    f.write(f'   - Total de amostras: {X.shape[0]}\n\n')
    
    f.write('3. DIVISÃO DOS DADOS\n')
    f.write(f'   - Treino: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)\n')
    f.write(f'   - Validação: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)\n')
    f.write(f'   - Teste: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)\n\n')
    
    f.write('4. DISTRIBUIÇÃO DAS CLASSES\n')
    f.write('   Treino:\n')
    for label, count in y_train.value_counts().items():
        pct = count / len(y_train) * 100
        f.write(f'     - {label}: {count} ({pct:.2f}%)\n')
    f.write('\n   Validação:\n')
    for label, count in y_val.value_counts().items():
        pct = count / len(y_val) * 100
        f.write(f'     - {label}: {count} ({pct:.2f}%)\n')
    f.write('\n   Teste:\n')
    for label, count in y_test.value_counts().items():
        pct = count / len(y_test) * 100
        f.write(f'     - {label}: {count} ({pct:.2f}%)\n')
    f.write('\n')
    
    f.write('5. ARQUIVOS GERADOS\n')
    f.write('   - data/processed/train.csv\n')
    f.write('   - data/processed/validation.csv\n')
    f.write('   - data/processed/test.csv\n\n')
    
    f.write('='*60 + '\n')
    f.write('Resumo gerado automaticamente pelo notebook 02\n')
    f.write('='*60 + '\n')

print(f'✓ Resumo do preprocessing salvo em: {summary_path}')


✓ Resumo do preprocessing salvo em: ../results/metrics/preprocessing_summary.txt


## 10. Resumo do Pré-processamento

**Etapas realizadas:**

1. Remoção de colunas de metadados
2. Tratamento de valores faltantes
3. Remoção de features com baixa variância
4. Normalização das features (StandardScaler)
5. Divisão dos dados (treino/validação/teste)
6. Salvamento dos dados processados

**Próximo passo:** Spot-checking de algoritmos

---